# 02 Data Warehouse Design

## Business Questions

### Overall Fatality Analysis
- What are the trends in road accident fatalities by year and month?
- Which days of the week have the highest number of fatalities?
- Which season has the highest average number of fatalities per year?

### Geographic Analysis
- What is the distribution of fatalities across regions?
- Which provinces and districts have the highest numbers of fatalities?
- Which provinces record the highest number of fatalities each year?

### Vehicle Analysis
- Which vehicle types are associated with the highest number of fatalities?

### Demographic Analysis
- Which age groups experience the highest number of fatalities?
- Which gender experiences the highest number of fatalities?

## Data Integration Requirements

### Geographic Reference Data

| Source | Purpose |
|---------|---------|
| Thailand Subdistrict Geographic Dataset | Standardize province, district, and subdistrict names, and enrich geographic attributes. |
| Province Abbreviation Reference | Map province abbreviations to official province names and ensure consistency across years. |

**Sources:**
- [ข้อมูลพิกัดตำบล](https://gdcatalognhic.nha.co.th/dataset/tambon/resource/d65b115f-8ee6-4381-8d6c-6efff538a1b7) 
- [รายชื่ออักษรย่อของจังหวัดในประเทศไทย](https://th.wikipedia.org/wiki/%E0%B8%A3%E0%B8%B2%E0%B8%A2%E0%B8%8A%E0%B8%B7%E0%B9%88%E0%B8%AD%E0%B8%AD%E0%B8%B1%E0%B8%81%E0%B8%A9%E0%B8%A3%E0%B8%A2%E0%B9%88%E0%B8%AD%E0%B8%82%E0%B8%AD%E0%B8%87%E0%B8%88%E0%B8%B1%E0%B8%87%E0%B8%AB%E0%B8%A7%E0%B8%B1%E0%B8%94%E0%B9%83%E0%B8%99%E0%B8%9B%E0%B8%A3%E0%B8%B0%E0%B9%80%E0%B8%97%E0%B8%A8%E0%B9%84%E0%B8%97%E0%B8%A2)

### Date Dimension Reference

| Source | Purpose |
|---------|---------|
| DimDate Reference Table | Generate the date dimension and enrich calendar attributes. |

**Source:**
- [dimdates.com](https://dimdates.com/)

**Additional Enrichment:**
- Add **Buddhist Era (B.E.)** year.
- Derive **Season** from the calendar date.

### ICD-10 code and referencing

| Source | Purpose |
|---------|---------|
| National Center for Health Statistics | Provides standardized ICD-10 classification codes and definitions for causes of death used to build and validate dim_cause_of_death |

**Source:**
- [ICD-10 Classification (NCHS, CDC)](https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Publications/ICD10/)

## Fact Identification

**Name:** fact_fatality

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| death_record_key | INT | No | Surrogate key to uniquely identify each deceased record | Be unique |
| death_date_key | INT | No | Foreign key referencing dim_date, represents date of death | Value should should in id of dim_date and should not contain null value |
| demographic_key | INT | No | Foreign key referencing dim_demographic, represents a person's demographic profile, including age group and gender | Value should should in id of dim_demographic and should not contain null value for null value or outlier map to -1 |
| nationality_key | INT | No | Foreign key referencing dim_nationality, representing a person's nationality based on the ISO 3166-1 standard. Unknown or unmapped nationalities are mapped to the 'Unknown' member in dim_nationality | Value should should in id of dim_nationality and should not contain null value for null value or unknown map to 99 |
| living_location_key | INT | No | Foreign key referencing dim_location, represents person's place of residence | Value should in id of dim_location |
| death_location_key | INT | No | Foreign key referencing dim_location, represents location where the person died | Value should in id of dim_location |
| vehicle_key | INT | No | Foreign key referencing dim_vehicle, representing the vehicle involved in the accident | Value should in id of dim_vehicle |
| cause_key | INT | No | Foreign key referencing dim_cause_of_death, representing the cause of death based on the ICD-10 classification | Value should in id of dim_cause_of_death |
| fatality_count | INT | No | Additive measure representing the number of fatalities | Default value = 1 |

**Grain:** One row in fact_fatality represents one deceased person resulting from a road traffic accident.


## Dimension Identification

**Name:** dim_date

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | Surrogate key to uniquely identify the date | Primary key. Stored in yyyyMMdd format (e.g., 20000101) |
| date | DATE | No | Full calendar date associated with the date dimension record| Must be in yyyy-MM-dd format (e.g., 2000-01-01) cannot be null. |
| date_description | string | No |Description of a date | example data 1 Jan 2000 cannot be null|
| day_fullname | string | No | Full name of a day | example data Saturday, cannot be null |
| day_shortname | string | No | Short name of a day | example data Sat, cannot be null |
| month_fullname | string | No | Full name of a month | example data  January, cannot be null |
| month_shortname | string | No | Short name of a month | example data  Jan, cannot be null |
| season | string | No | Season based on Thai meteorological seasons | (Summer, Rainy, Winter), cannot be null |
| calendar_day | INT | No | The day number in the calendar year | day order in year start at 1 Jan |
| calendar_day_in_week | INT | No | The day number in the week. | counting start from Sunday |
| calendar_month | INT | No | The calendar month number in the year. | counting month start at Jan |
| calendar_day_in_month | INT | No | The day number in the calendar month. | day number |
| calendar_quarter | INT | No | The calendar quarter number in the year. | quarter number (1,2,3,4) |
| calendar_day_in_quarter | INT | No |	The day number in the calendar quarter. | counting of day start at first day of quarter |
| calendar_day_in_season | INT | No |	The day number in the season. | counting of day start at first day of season |
| calendar_year | INT | No | The calendar year number. | numerical year in A.D. |
| calendar_year_be | INT | No | The calendar year number in B.E. | numerical year in B.E. (A.D. + 543) |


**Name:** dim_demographic

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | Surrogate key to uniquely identify demographic group | Primary Key, rows size = 6 * 3 = 18 |
| age_from | INT | Yes | start age of age group | Null indicate no lower bound |
| age_to | INT | Yes | Til age of age group | Null indicate upper bound |
| age_group | string | No | Name group age group | Must be one of: <br> **Toddler** (0 - 5 y), **Kids** (6 - 13 y), **Teenage** (14 - 24 y), **Adult** (25 - 64 y), **Older** (65 - 110 y), **Unknown** (-infinity - 0 & 111 - infinity y)|
| sex | string | No | Gender | (Male, Female, Unknown) |

**Name:** dim_nationality

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | Surrogate key to uniquely identify the nationality, 99 is unknown value. Base on iso 3166-1 | Value base on ISO 3166-1 numerical, mask unknown as 99 |
| nationality_name | string | No | Name of nationality | Nationality name base on ISO 3166-1 |
| nationality_2_alpha | string | No | 2 alphabet name | Nationality name base on ISO 3166-1 2 alpha |
| nationality_3_alpha | string | No | 3 alphabet name | Nationality name base on ISO 3166-1 3 alpha |

**Name:** dim_location

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | Surrogate key to uniquely identify location | Primary Key |
| subdistrict | string | No | Subdistrict name in english, store N/A as Unknown | not null map N/A to be unknown |
| district | string | No | District name in english, store N/A as Unknown | not null map N/A to be unknown |
| province | string | No | Province name in english, store N/A as Unknown | not null map N/A to be unknown |
| subdistrict_th | string | No | Subdistrict name in thai, store N/A as Unknown | not null map N/A to be unknown |
| district_th | string | No | District name in thai, store N/A as Unknown | not null map N/A to be unknown |
| province_th | string | No | Province name in thai, store N/A as Unknown | not null map N/A to be unknown |
| province_short_name_th | string | No | Province name in thai, store N/A as Unknown | not null map N/A to be unknown |
| province_lat | float | Yes | latitude | use province latitude |
| province_lon | float | Yes | longitude | use province longitude |

**Name:** dim_vehicle

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | Surrogate key to uniquely identify vehicle | Primary Key |
| vehicle_type_th | string | No | vehicle type in thai, N/A with unknown | not null map N/A to be unknown  |
| vehicle_type_en | string | No | vehicle type in english, N/A with unknown | not null map N/A to be unknown  |

**Name:** dim_cause_of_death

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | Surrogate key to uniquely identify  cause | Primary Key |
| code | string | No | ICD-10 code, 1 field for unknown | ICD-10 code UU for unknown |
| description | string | No | ICD-10 description | description base on code |

# Schema and Data model

![schema](../Docs/data_model.png)